# Pricing: MiniLM + Cleanlab + LightGBM

Full eligible-data training. MiniLM learns description-to-log-price regression with OOF predictions. Cleanlab audits observed training log prices using structured vehicle facts, internal CV and uncertainty estimation; text-price features are excluded from the audit. Huber LightGBM uses retained rows. Validation/test are untouched. Make/model remain categorical. Choose a CUDA runtime and a fresh Cleanlab output directory. Installed legacy weights and metrics are unchanged.


In [ ]:
!git clone https://github.com/botsandbolts/Pricing.git /content/pricing
%cd /content/pricing
!pip -q install -r requirements-training.txt
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime"


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
source=Path("/content/vehicles.zip")
if not source.exists():
    urlretrieve("https://zenodo.org/records/10457828/files/Used%20Cars%20Dataset.zip?download=1",source)
!python -m scripts.prepare_full /content/vehicles.zip --output training-cache/full-cohort.jsonl


In [ ]:
!python -m pricing.train --data training-cache/full-cohort.jsonl --output artifacts/car-price-studio-cleanlab --device cuda --folds 3 --epochs 2
assert Path("artifacts/car-price-studio-cleanlab/manifest.json").exists(), "Training did not finish; inspect the log"


Evaluation uses chronological training OOF prices and train-only models for validation/test. Only after scoring does a fresh OOF pass across all eligible database rows and the Cleanlab label audit train the deployment stack. The assessment and all-row OOF CSVs record the fold for every prediction. Completed fold caches resume only with matching data/configuration. Final all-row models have no independent test set; reported scores belong to assessment.


In [ ]:
import json,shutil
print(json.dumps(json.loads(Path("artifacts/car-price-studio-cleanlab/evaluation.json").read_text()),indent=2))
archive=shutil.make_archive("/content/pricing-minilm-oof-all","zip","artifacts")
from google.colab import files
files.download(archive)
